# AMEX Enterprise Credit Risk Platform
## Notebook 23 — Phase 1, Problem 2: Risk Tier Classification — Ongoing Monitoring
### Problem Statement 2 of 14: Risk Tier Classification

CRISP-DM stage: **Deployment / Monitoring**. Depends on Notebook 20's real `risk_tier_assignments.csv` (hard dependency). Reuses Notebook 12's real simulated-monitoring-window pattern, applied to tier population and bad-rate stability instead of raw feature PSI.

**Honest framing, reused verbatim from Notebook 12.** The engineered feature store does not retain a per-customer calendar date, and there is no live production traffic to monitor. This notebook partitions the real, held-out scored population -- never trained on -- into sequential row-order batches as an **illustrative stand-in** for successive scoring runs. Every metric computed per window is a real, live computation on real data; only the "these arrived over time" framing is simulated, and every chart/table/report section repeats this label so it is never mistaken for genuine production telemetry.

**What this notebook does, all real:**

- Partitions the real risk-tier assignments into simulated monitoring windows (same adaptive sizing as Notebook 12).
- Per window: tier population stability (PSI vs. the full-population baseline), per-tier bad-rate drift, and rank-ordering (AUC) -- alerts built from Notebook 19's real KPI targets.
- A cross-tab consistency check between the quantile and business-rule tier assignments -- Cohen's kappa agreement, a Problem-2-specific addition beyond what Notebook 12 covers.
- A real, schedulable `risk_tier_monitoring_job.py`, generated the same way as Notebook 12's `monitoring_job.py`.

**Deliverables:** `risk_tier_monitoring_windows_report.csv`, `risk_tier_alert_log.csv`, `risk_tier_monitoring_baseline.json`, `risk_tier_monitoring_job.py`, charts, `Risk_Tier_Monitoring_Report.docx`, `notebook_23_summary.json`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG & NOTEBOOK 19/20 REAL OUTPUTS
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config & Notebook 19/20 Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB20_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_20_summary.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first.")
if not NB20_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"{NB20_SUMMARY_PATH} not found.\nNotebook 23 has a hard dependency on Notebook 20's real risk-tier "
        f"assignments -- fix: run 20_risk_tier_model_development.ipynb first."
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB20_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB20_SUMMARY = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (_resource_limits.get("warp_thread_count") or PROJECT_CONFIG.get("warp_thread_count")
                      or PROJECT_CONFIG["hardware"].get("logical_cores_detected"))

_REQUIRED_PILLARS = {
    "risk_tier_policy": "02_Problem2_Risk_Tier_Classification/01_Risk_Tier_Policy",
    "risk_tier_modeling": "02_Problem2_Risk_Tier_Classification/02_Risk_Tier_Modeling",
    "risk_tier_validation": "02_Problem2_Risk_Tier_Classification/03_Risk_Tier_Validation",
    "risk_tier_deployment": "02_Problem2_Risk_Tier_Classification/04_Risk_Tier_Deployment",
    "risk_tier_monitoring": "02_Problem2_Risk_Tier_Classification/05_Risk_Tier_Monitoring",
    "risk_tier_reporting": "02_Problem2_Risk_Tier_Classification/06_Risk_Tier_Reporting",
    "risk_tier_packaging": "02_Problem2_Risk_Tier_Classification/07_Risk_Tier_Packaging",
}
_config_healed = False
for _key, _rel_path in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _rel_path
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json -- added automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print("\u2705 project_config.json updated in place -- no need to re-run Notebook 01.")

RISK_TIER_MONITORING_DIR = PILLAR_DIRS["risk_tier_monitoring"]
RISK_TIER_MONITORING_DIR.mkdir(parents=True, exist_ok=True)

ASSIGNMENTS_PATH = Path(NB20_SUMMARY["output_files"]["risk_tier_assignments.csv"])
if not ASSIGNMENTS_PATH.exists():
    raise FileNotFoundError(f"{ASSIGNMENTS_PATH} not found.\nFix: re-run 20_risk_tier_model_development.ipynb.")

CHAMPION_NAME = NB20_SUMMARY["champion_model"]
PRIMARY_METHOD = NB20_SUMMARY["primary_method"]
N_TIERS = NB20_SUMMARY["n_tiers"]

print(f"Champion model (Problem 1, real)     : {CHAMPION_NAME}")
print(f"Primary tiering method (Notebook 19)  : {PRIMARY_METHOD}")
print(f"Monitoring artifacts will be written under: {RISK_TIER_MONITORING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from sklearn.metrics import roc_auc_score, cohen_kappa_score
except ImportError:
    missing.append("scikit-learn")
try:
    from docx import Document
    from docx.shared import Inches
    from docx.enum.text import WD_ALIGN_PARAGRAPH
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) + "\n"
                       f"Fix: pip install {' '.join(missing)}")

logger.info(f"Thread pool configured to {WARP_THREAD_COUNT} threads (95% cap, WARP 6.4, Concurrency)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)

print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: LOAD REAL ASSIGNMENTS & POLICY, PARTITION INTO SIMULATED WINDOWS
# =============================================================================
_section("SECTION 3: Load Real Assignments & Policy, Partition Into Simulated Windows")

assignments_df = pd.read_csv(ASSIGNMENTS_PATH)
RISK_TIER_POLICY_PATH = PILLAR_DIRS["risk_tier_policy"] / "risk_tier_policy.json"
with open(RISK_TIER_POLICY_PATH, "r", encoding="utf-8") as f:
    RISK_TIER_POLICY = json.load(f)
TIER_ORDER = RISK_TIER_POLICY["tier_order"]
KPI_TARGETS = RISK_TIER_POLICY["kpi_targets"]

# --- Same illustrative row-order partition as Notebook 12; see the notebook
#     intro's honesty note. ---
N_WINDOWS_REQUESTED = 6
MIN_WINDOW_SIZE = 300
_n = len(assignments_df)
N_MONITORING_WINDOWS = max(1, min(N_WINDOWS_REQUESTED, _n // MIN_WINDOW_SIZE)) if _n >= MIN_WINDOW_SIZE else 1
_window_index_splits = np.array_split(np.arange(_n), N_MONITORING_WINDOWS)

BASELINE_BAD_RATE = float(assignments_df["actual_default"].mean())
BASELINE_TIER_DIST = (assignments_df["risk_tier_primary"].value_counts().reindex(TIER_ORDER).fillna(0) / _n)

print(f"Scored population: {_n:,} customers (real, held-out)")
print(f"SIMULATED into {N_MONITORING_WINDOWS} sequential monitoring window(s) of ~{_n // N_MONITORING_WINDOWS:,} "
      f"customers each (row-order partition -- illustrative only, see notebook intro).")
print(f"Baseline bad rate (full population): {BASELINE_BAD_RATE:.4%}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: TIER POPULATION STABILITY (PSI) PER MONITORING WINDOW
# =============================================================================
_section("SECTION 4: Tier Population Stability (PSI) Per Monitoring Window")


def _tier_pct_dist(tier_assignments):
    _counts = pd.Series(tier_assignments).value_counts().reindex(TIER_ORDER).fillna(0)
    return (_counts / max(_counts.sum(), 1)).clip(lower=1e-4)


_psi_by_window = []
for _widx, _ridx in enumerate(_window_index_splits):
    _window_num = _widx + 1
    _window_tiers = assignments_df["risk_tier_primary"].to_numpy()[_ridx]
    _window_dist = _tier_pct_dist(_window_tiers)
    _baseline_clipped = BASELINE_TIER_DIST.clip(lower=1e-4)
    _psi = float(((_window_dist - _baseline_clipped) * np.log(_window_dist / _baseline_clipped)).sum())
    _psi_by_window.append({"window": _window_num, "n_customers": len(_ridx), "tier_population_psi": round(_psi, 5)})

psi_by_window_df = pd.DataFrame(_psi_by_window)
print(psi_by_window_df.to_string(index=False))
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: PER-TIER BAD-RATE DRIFT & RANK-ORDERING (AUC) PER WINDOW
# =============================================================================
_section("SECTION 5: Per-Tier Bad-Rate Drift & Rank-Ordering (AUC) Per Window")

_perf_by_window = []
for _widx, _ridx in enumerate(_window_index_splits):
    _window_num = _widx + 1
    _y_w = assignments_df["actual_default"].to_numpy()[_ridx]
    _pd_w = assignments_df["predicted_pd"].to_numpy()[_ridx]
    _window_bad_rate = float(_y_w.mean())
    _delta_pp = (_window_bad_rate - BASELINE_BAD_RATE) * 100.0
    _window_auc = float(roc_auc_score(_y_w, _pd_w)) if len(np.unique(_y_w)) >= 2 else None
    _perf_by_window.append({
        "window": _window_num, "n_customers": len(_ridx),
        "actual_bad_rate": round(_window_bad_rate, 5),
        "avg_predicted_pd": round(float(_pd_w.mean()), 5),
        "delta_vs_baseline_pp": round(_delta_pp, 3),
        "auc": round(_window_auc, 5) if _window_auc is not None else None,
    })

perf_by_window_df = pd.DataFrame(_perf_by_window)
print(f"Baseline (full-population) bad rate: {BASELINE_BAD_RATE:.4%}")
print(perf_by_window_df.to_string(index=False))
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: CROSS-TAB CONSISTENCY -- QUANTILE VS. BUSINESS-RULE AGREEMENT
# =============================================================================
_section("SECTION 6: Cross-Tab Consistency -- Quantile vs. Business-Rule Agreement")

# --- Problem-2-specific addition: how often do the two independently-computed
#     tier assignments (Notebook 20) agree on the same customer? Cohen's kappa
#     accounts for chance agreement, unlike raw percent-agreement. ---
_cross_tab = pd.crosstab(assignments_df["risk_tier_business_rule"], assignments_df["risk_tier_quantile"])
_cross_tab = _cross_tab.reindex(index=TIER_ORDER, columns=TIER_ORDER, fill_value=0)
_raw_agreement = float((assignments_df["risk_tier_business_rule"] == assignments_df["risk_tier_quantile"]).mean())
_kappa = float(cohen_kappa_score(assignments_df["risk_tier_business_rule"], assignments_df["risk_tier_quantile"]))

print(_cross_tab.to_string())
print(f"\nRaw agreement rate       : {_raw_agreement:.4%}")
print(f"Cohen's kappa             : {_kappa:.4f}  "
      f"({'poor' if _kappa < 0.2 else 'fair' if _kappa < 0.4 else 'moderate' if _kappa < 0.6 else 'substantial' if _kappa < 0.8 else 'almost perfect'} agreement)")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: APPLY MONITORING THRESHOLDS & BUILD THE ALERT LOG
# =============================================================================
_section("SECTION 7: Apply Monitoring Thresholds & Build the Alert Log")

# --- Thresholds reused from Notebook 19's real, saved KPI targets where
#     available (max_tier_population_psi_split_half); psi_significant_shift
#     and min_acceptable_auc follow Notebook 12's own convention -- clearly
#     labeled ASSUMPTION, editable. ---
MONITORING_THRESHOLDS = {
    "tier_population_psi_moderate": KPI_TARGETS["max_tier_population_psi_split_half"],   # Notebook 19 policy
    "tier_population_psi_significant": KPI_TARGETS["max_tier_population_psi_split_half"] * 2.5,  # ASSUMPTION
    "bad_rate_swing_pp": 5.0,          # ASSUMPTION -- consistent with Notebook 12's own default_rate_swing_pp
    "min_acceptable_auc": 0.70,        # ASSUMPTION -- consistent with Notebook 12
    "min_kappa_agreement": 0.60,       # ASSUMPTION -- "substantial agreement" per Landis & Koch (1977)
}

_alert_rows = []
for _row in _psi_by_window:
    _w = _row["window"]
    if _row["tier_population_psi"] >= MONITORING_THRESHOLDS["tier_population_psi_significant"]:
        _status = "ALERT"
    elif _row["tier_population_psi"] >= MONITORING_THRESHOLDS["tier_population_psi_moderate"]:
        _status = "WATCH"
    else:
        _status = "OK"
    _alert_rows.append({"window": _w, "metric": "tier_population_psi", "value": _row["tier_population_psi"],
                         "threshold": MONITORING_THRESHOLDS["tier_population_psi_significant"], "status": _status,
                         "detail": "tier population share vs. full-population baseline"})

for _row in _perf_by_window:
    _w = _row["window"]
    _abs_delta = abs(_row["delta_vs_baseline_pp"])
    _status = "ALERT" if _abs_delta >= MONITORING_THRESHOLDS["bad_rate_swing_pp"] else (
        "WATCH" if _abs_delta >= MONITORING_THRESHOLDS["bad_rate_swing_pp"] / 2 else "OK")
    _alert_rows.append({"window": _w, "metric": "bad_rate_swing_pp", "value": _row["delta_vs_baseline_pp"],
                         "threshold": MONITORING_THRESHOLDS["bad_rate_swing_pp"], "status": _status,
                         "detail": f"actual {_row['actual_bad_rate']:.4%} vs. baseline {BASELINE_BAD_RATE:.4%}"})
    if _row["auc"] is None:
        _alert_rows.append({"window": _w, "metric": "rank_ordering_auc", "value": None,
                             "threshold": MONITORING_THRESHOLDS["min_acceptable_auc"], "status": "NOT_COMPUTABLE",
                             "detail": "window has only one outcome class -- AUC undefined this window"})
    else:
        _status = "ALERT" if _row["auc"] < MONITORING_THRESHOLDS["min_acceptable_auc"] else "OK"
        _alert_rows.append({"window": _w, "metric": "rank_ordering_auc", "value": _row["auc"],
                             "threshold": MONITORING_THRESHOLDS["min_acceptable_auc"], "status": _status,
                             "detail": f"AUC = {_row['auc']:.4f}"})

_kappa_status = "OK" if _kappa >= MONITORING_THRESHOLDS["min_kappa_agreement"] else "WATCH"
_alert_rows.append({"window": "all", "metric": "quantile_vs_business_rule_kappa", "value": round(_kappa, 4),
                     "threshold": MONITORING_THRESHOLDS["min_kappa_agreement"], "status": _kappa_status,
                     "detail": f"raw agreement {_raw_agreement:.4%}"})

alert_log_df = pd.DataFrame(_alert_rows)
alert_log_path = RISK_TIER_MONITORING_DIR / "risk_tier_alert_log.csv"
alert_log_df.to_csv(alert_log_path, index=False)

_n_alerts = int((alert_log_df["status"] == "ALERT").sum())
_n_watch = int((alert_log_df["status"] == "WATCH").sum())
print(f"Alert log: {len(alert_log_df)} checks across {N_MONITORING_WINDOWS} window(s) -- "
      f"{_n_alerts} ALERT, {_n_watch} WATCH")
print(f"\u2705 Saved -> {alert_log_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: MONITORING WINDOWS REPORT (COMBINED TABLE)
# =============================================================================
_section("SECTION 8: Monitoring Windows Report (Combined Table)")

monitoring_windows_df = psi_by_window_df.merge(perf_by_window_df, on=["window", "n_customers"])
monitoring_windows_path = RISK_TIER_MONITORING_DIR / "risk_tier_monitoring_windows_report.csv"
monitoring_windows_df.to_csv(monitoring_windows_path, index=False)
print(monitoring_windows_df.to_string(index=False))
print(f"\u2705 Saved -> {monitoring_windows_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: MONITORING BASELINE, CONFIG & A REAL, SCHEDULABLE risk_tier_monitoring_job.py
# =============================================================================
_section("SECTION 9: Monitoring Baseline, Config & a Real, Schedulable risk_tier_monitoring_job.py")

monitoring_baseline = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_model": CHAMPION_NAME,
    "primary_method": PRIMARY_METHOD,
    "tier_order": TIER_ORDER,
    "baseline_tier_distribution": {t: float(BASELINE_TIER_DIST[t]) for t in TIER_ORDER},
    "baseline_bad_rate": BASELINE_BAD_RATE,
    "n_baseline_rows": int(_n),
}
monitoring_baseline_path = RISK_TIER_MONITORING_DIR / "risk_tier_monitoring_baseline.json"
with open(monitoring_baseline_path, "w", encoding="utf-8") as f:
    json.dump(monitoring_baseline, f, indent=2)
print(f"\u2705 Saved -> {monitoring_baseline_path}")

monitoring_config = {
    "thresholds": MONITORING_THRESHOLDS,
    "check_cadence": "ASSUMPTION -- recommended: run risk_tier_monitoring_job.py against each new scored batch "
                      "(daily batch cadence assumed); review the trend report weekly regardless of alert status.",
    "escalation_policy": "ASSUMPTION -- route ALERT-status findings to the Model Risk Management team "
                          "(Notebook 07 / Notebook 21) within 1 business day; WATCH-status findings are logged "
                          "and reviewed at the next scheduled tier-policy review.",
    "notes": "tier_population_psi_moderate is reused verbatim from Notebook 19's real KPI target "
             "(max_tier_population_psi_split_half); all other thresholds are this notebook's own editable "
             "ASSUMPTION, consistent with Notebook 12's convention.",
}
monitoring_config_path = RISK_TIER_MONITORING_DIR / "risk_tier_monitoring_config.json"
with open(monitoring_config_path, "w", encoding="utf-8") as f:
    json.dump(monitoring_config, f, indent=2)
print(f"\u2705 Saved -> {monitoring_config_path}")

# --- Generated via plain "\n".join([...]) -- this platform's established
#     convention for generated source (Notebook 10's main.py, Notebook 12's
#     monitoring_job.py). Honest scope note, same as Notebook 12: there is no
#     real "next batch" file to feed this job today, so it is validated by
#     compiling its real source (Section 11), not by executing it against
#     live data. ---
MONITORING_JOB_SOURCE = "\n".join([
    "# AMEX Enterprise Credit Risk Platform -- Scheduled Risk-Tier Monitoring Job.",
    "# Auto-generated by 23_risk_tier_monitoring.ipynb. Intended usage (e.g. a daily",
    "# cron job or Windows Task Scheduler task):",
    "#     python risk_tier_monitoring_job.py --new-data-csv path/to/new_scored_batch.csv",
    "# Exits 0 if all checks are OK/WATCH, exits 1 if any check is ALERT.",
    "import argparse",
    "import csv",
    "import json",
    "import sys",
    "from datetime import datetime, timezone",
    "from pathlib import Path",
    "",
    "import numpy as np",
    "",
    "HERE = Path(__file__).resolve().parent",
    "",
    "",
    "def main():",
    "    parser = argparse.ArgumentParser(description=\"AMEX risk-tier scheme production monitoring job\")",
    "    parser.add_argument(\"--new-data-csv\", required=True,",
    "                         help=\"Path to a new scored batch CSV with columns: risk_tier_primary, actual_default (optional)\")",
    "    parser.add_argument(\"--baseline-json\", default=str(HERE / \"risk_tier_monitoring_baseline.json\"))",
    "    parser.add_argument(\"--config-json\", default=str(HERE / \"risk_tier_monitoring_config.json\"))",
    "    parser.add_argument(\"--out-log\", default=str(HERE / \"risk_tier_monitoring_job_log.csv\"))",
    "    args = parser.parse_args()",
    "",
    "    with open(args.baseline_json, \"r\", encoding=\"utf-8\") as f:",
    "        baseline = json.load(f)",
    "    with open(args.config_json, \"r\", encoding=\"utf-8\") as f:",
    "        config = json.load(f)",
    "    thresholds = config[\"thresholds\"]",
    "    tier_order = baseline[\"tier_order\"]",
    "",
    "    import pandas as pd",
    "    new_df = pd.read_csv(args.new_data_csv)",
    "",
    "    result = {\"run_at_utc\": datetime.now(timezone.utc).isoformat(), \"new_data_csv\": args.new_data_csv,",
    "              \"n_rows\": len(new_df), \"checks\": []}",
    "    alert = False",
    "",
    "    if \"risk_tier_primary\" in new_df.columns:",
    "        counts = new_df[\"risk_tier_primary\"].value_counts().reindex(tier_order).fillna(0)",
    "        dist = (counts / max(counts.sum(), 1)).clip(lower=1e-4)",
    "        baseline_dist = pd.Series(baseline[\"baseline_tier_distribution\"]).reindex(tier_order).clip(lower=1e-4)",
    "        psi = float(((dist - baseline_dist) * np.log(dist / baseline_dist)).sum())",
    "        status = (\"ALERT\" if psi >= thresholds[\"tier_population_psi_significant\"]",
    "                  else \"WATCH\" if psi >= thresholds[\"tier_population_psi_moderate\"] else \"OK\")",
    "        alert = alert or (status == \"ALERT\")",
    "        result[\"checks\"].append({\"metric\": \"tier_population_psi\", \"value\": round(psi, 5), \"status\": status})",
    "    else:",
    "        result[\"checks\"].append({\"metric\": \"tier_population_psi\", \"value\": None,",
    "                                  \"status\": \"NOT_COMPUTABLE\", \"detail\": \"no 'risk_tier_primary' column\"})",
    "",
    "    if \"actual_default\" in new_df.columns:",
    "        bad_rate = float(new_df[\"actual_default\"].mean())",
    "        delta_pp = (bad_rate - baseline[\"baseline_bad_rate\"]) * 100.0",
    "        status = \"ALERT\" if abs(delta_pp) >= thresholds[\"bad_rate_swing_pp\"] else \"OK\"",
    "        alert = alert or (status == \"ALERT\")",
    "        result[\"checks\"].append({\"metric\": \"bad_rate_swing_pp\", \"value\": round(delta_pp, 3), \"status\": status})",
    "    else:",
    "        result[\"checks\"].append({\"metric\": \"bad_rate_swing_pp\", \"value\": None,",
    "                                  \"status\": \"NOT_COMPUTABLE\", \"detail\": \"no 'actual_default' column -- outcomes not yet realized\"})",
    "",
    "    write_header = not Path(args.out_log).exists()",
    "    with open(args.out_log, \"a\", encoding=\"utf-8\", newline=\"\") as f:",
    "        writer = csv.writer(f)",
    "        if write_header:",
    "            writer.writerow([\"run_at_utc\", \"new_data_csv\", \"n_rows\", \"any_alert\"])",
    "        writer.writerow([result[\"run_at_utc\"], result[\"new_data_csv\"], result[\"n_rows\"], alert])",
    "",
    "    print(json.dumps(result, indent=2))",
    "    sys.exit(1 if alert else 0)",
    "",
    "",
    "if __name__ == \"__main__\":",
    "    main()",
    "",
])

monitoring_job_path = RISK_TIER_MONITORING_DIR / "risk_tier_monitoring_job.py"
with open(monitoring_job_path, "w", encoding="utf-8") as f:
    f.write(MONITORING_JOB_SOURCE)
print(f"\u2705 Saved -> {monitoring_job_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: MONITORING READINESS CHECKLIST
# =============================================================================
_section("SECTION 10: Monitoring Readiness Checklist")

readiness_checklist = [
    {"dimension": "Monitoring Windows Computed", "status": "Pass", "evidence": f"{N_MONITORING_WINDOWS} window(s) (this run)"},
    {"dimension": "Tier Population Stability Tracked", "status": "Pass" if _n_alerts == 0 else "Review Needed",
     "evidence": f"{_n_alerts} ALERT(s), {_n_watch} WATCH across {len(alert_log_df)} checks"},
    {"dimension": "Bad-Rate Drift Tracked", "status": "Pass", "evidence": "per-window swing vs. baseline (this run)"},
    {"dimension": "Rank-Ordering (AUC) Tracked", "status": "Pass", "evidence": "per-window AUC (this run)"},
    {"dimension": "Method Consistency (Kappa) Tracked", "status": _kappa_status,
     "evidence": f"kappa={_kappa:.4f} (this run)"},
    {"dimension": "Scheduled Job Generated", "status": "Pass", "evidence": monitoring_job_path.name},
    {"dimension": "Baseline & Config Persisted", "status": "Pass",
     "evidence": f"{monitoring_baseline_path.name}, {monitoring_config_path.name}"},
]
readiness_df = pd.DataFrame(readiness_checklist)
readiness_path = RISK_TIER_MONITORING_DIR / "risk_tier_monitoring_readiness_checklist.csv"
readiness_df.to_csv(readiness_path, index=False)
print(readiness_df.to_string(index=False))
print(f"\u2705 Saved -> {readiness_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: SYNTAX SELF-CHECK ON THE GENERATED MONITORING JOB
# =============================================================================
_section("SECTION 11: Syntax Self-Check on the Generated Monitoring Job")

compile(MONITORING_JOB_SOURCE, str(monitoring_job_path), "exec")
print(f"risk_tier_monitoring_job.py compiles cleanly ({len(MONITORING_JOB_SOURCE.splitlines())} lines).")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: CHARTS
# =============================================================================
_section("SECTION 12: Charts")

VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_red": "#e34948", "cat_green": "#3a9e5f", "cat_amber": "#d69a2a"}
PROBLEM_NAME = "Phase 1 \u00b7 Problem 2 -- Risk Tier Classification"


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"]); ax.figure.set_facecolor(VIZ["surface"])
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.8, zorder=0); ax.set_axisbelow(True)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(VIZ["grid"])
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=9)
    ax.title.set_color(VIZ["text_primary"])


# Chart 1: bad rate trend across simulated windows, with baseline line
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
ax.plot(perf_by_window_df["window"], perf_by_window_df["actual_bad_rate"] * 100, marker="o",
        color=VIZ["cat_blue"], linewidth=2, zorder=3, label="Actual bad rate")
ax.axhline(BASELINE_BAD_RATE * 100, color=VIZ["cat_red"], linestyle="--", linewidth=1.5, label="Baseline (full population)")
_style_axes(ax)
ax.set_xlabel("Simulated monitoring window"); ax.set_ylabel("Bad rate (%)")
ax.set_title(f"{PROBLEM_NAME}\nBad Rate by Simulated Window vs. Baseline (Real, Measured -- Illustrative Windows)", fontsize=10)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart1_path = RISK_TIER_MONITORING_DIR / "bad_rate_trend_chart.png"
fig.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart1_path}")

# Chart 2: tier population PSI by window
fig, ax = plt.subplots(figsize=(8, 5.5), dpi=150)
_bars = ax.bar(psi_by_window_df["window"], psi_by_window_df["tier_population_psi"], color=VIZ["cat_amber"], zorder=3)
ax.axhline(MONITORING_THRESHOLDS["tier_population_psi_significant"], color=VIZ["cat_red"], linestyle="--",
           linewidth=1.5, label="Significant-shift threshold")
ax.bar_label(_bars, padding=3, fontsize=9, fmt="%.4f")
_style_axes(ax)
ax.set_xlabel("Simulated monitoring window"); ax.set_ylabel("Tier population PSI")
ax.set_title(f"{PROBLEM_NAME}\nTier Population Stability by Simulated Window (Real, Measured)", fontsize=11)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
chart2_path = RISK_TIER_MONITORING_DIR / "tier_population_psi_chart.png"
fig.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart2_path}")

# Chart 3: cross-tab agreement heatmap
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=150)
_im = ax.imshow(_cross_tab.values, cmap="Blues")
ax.set_xticks(range(len(TIER_ORDER))); ax.set_xticklabels(TIER_ORDER, rotation=30, ha="right")
ax.set_yticks(range(len(TIER_ORDER))); ax.set_yticklabels(TIER_ORDER)
ax.set_xlabel("Quantile method"); ax.set_ylabel("Business-rule method")
for _i in range(len(TIER_ORDER)):
    for _j in range(len(TIER_ORDER)):
        ax.text(_j, _i, str(_cross_tab.values[_i, _j]), ha="center", va="center",
                color="white" if _cross_tab.values[_i, _j] > _cross_tab.values.max() / 2 else "black", fontsize=9)
ax.set_title(f"{PROBLEM_NAME}\nMethod Agreement -- Business-Rule vs. Quantile (Real, kappa={_kappa:.3f})", fontsize=10)
fig.colorbar(_im, ax=ax, shrink=0.8, label="Customers")
fig.tight_layout()
chart3_path = RISK_TIER_MONITORING_DIR / "method_agreement_heatmap_chart.png"
fig.savefig(chart3_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {chart3_path}")

print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WORD REPORT -- RISK_TIER_MONITORING_REPORT.DOCX
# =============================================================================
_section("SECTION 13: Word Report -- Risk_Tier_Monitoring_Report.docx")

doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 1, Problem 2: Risk Tier Classification -- Ongoing Monitoring Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

doc.add_heading("1. Honest Framing", level=1)
doc.add_paragraph(
    "There is no live production traffic or per-customer calendar date available at this platform's current "
    "aggregation stage. This report partitions the real, held-out scored population into sequential row-order "
    "batches as an illustrative stand-in for successive scoring runs -- every metric below is a real, live "
    "computation on real data; only the 'these arrived over time' framing is simulated."
)

doc.add_heading("2. Monitoring Windows Report", level=1)
_t = doc.add_table(rows=1, cols=len(monitoring_windows_df.columns))
_t.style = "Light Grid Accent 1"
for _i, _col in enumerate(monitoring_windows_df.columns):
    _t.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in monitoring_windows_df.iterrows():
    _cells = _t.add_row().cells
    for _i, _col in enumerate(monitoring_windows_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("3. Alert Log", level=1)
_t2 = doc.add_table(rows=1, cols=len(alert_log_df.columns))
_t2.style = "Light Grid Accent 1"
for _i, _col in enumerate(alert_log_df.columns):
    _t2.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in alert_log_df.iterrows():
    _cells = _t2.add_row().cells
    for _i, _col in enumerate(alert_log_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("4. Method Consistency", level=1)
doc.add_paragraph(
    f"Business-rule vs. quantile tier assignment: raw agreement {_raw_agreement:.4%}, Cohen's kappa {_kappa:.4f}. "
    f"A low kappa indicates the two methods disagree materially on individual customers even where their tier "
    f"population shares look similar in aggregate -- worth investigating before using both interchangeably."
)

doc.add_heading("5. Monitoring Readiness Checklist", level=1)
_t3 = doc.add_table(rows=1, cols=len(readiness_df.columns))
_t3.style = "Light Grid Accent 1"
for _i, _col in enumerate(readiness_df.columns):
    _t3.rows[0].cells[_i].text = _col.replace("_", " ").title()
for _, _row in readiness_df.iterrows():
    _cells = _t3.add_row().cells
    for _i, _col in enumerate(readiness_df.columns):
        _cells[_i].text = str(_row[_col])

doc.add_heading("6. Charts", level=1)
for _cp, _cap in [(chart1_path, "Bad rate trend by simulated window"), (chart2_path, "Tier population PSI by window"),
                   (chart3_path, "Method agreement heatmap")]:
    doc.add_picture(str(_cp), width=Inches(6.0))
    _p = doc.add_paragraph(_cap); _p.alignment = WD_ALIGN_PARAGRAPH.CENTER

report_path = RISK_TIER_MONITORING_DIR / "Risk_Tier_Monitoring_Report.docx"
doc.save(report_path)
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 14: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Monitoring windows report covers all windows", len(monitoring_windows_df) == N_MONITORING_WINDOWS,
       f"({len(monitoring_windows_df)} vs {N_MONITORING_WINDOWS})")
_check("Alert log is non-empty", len(alert_log_df) > 0)
_check("Cross-tab sums to real assignment row count", int(_cross_tab.values.sum()) == len(assignments_df))
_check("Readiness checklist covers 7 dimensions", len(readiness_df) == 7, f"({len(readiness_df)})")
_check("Generated monitoring job compiles", True)  # Section 11 would have raised already if not

_expected_files = [monitoring_windows_path, alert_log_path, monitoring_baseline_path, monitoring_config_path,
                    monitoring_job_path, readiness_path, chart1_path, chart2_path, chart3_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 23 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 23 checks passed.")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 15: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "n_monitoring_windows": N_MONITORING_WINDOWS,
}
performance_report_path = ARTIFACTS_DIR / "notebook_23_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 15 complete.")


# =============================================================================
# SECTION 16: WRITE NOTEBOOK 23 SUMMARY ARTIFACT (for Notebook 24's rollup)
# =============================================================================
_section("SECTION 16: Write Notebook 23 Summary Artifact")

notebook_23_summary = {
    "notebook": "23_risk_tier_monitoring",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 2,
    "problem_name": "Risk Tier Classification",
    "champion_model": CHAMPION_NAME,
    "n_monitoring_windows": N_MONITORING_WINDOWS,
    "n_alerts": _n_alerts,
    "n_watch": _n_watch,
    "method_agreement_kappa": round(_kappa, 4),
    "method_raw_agreement": round(_raw_agreement, 4),
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb23_summary_path = ARTIFACTS_DIR / "notebook_23_summary.json"
with open(nb23_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_23_summary, f, indent=2)
print(f"\u2705 Saved -> {nb23_summary_path}")
print("\n\u2705 Section 16 complete.")


# =============================================================================
# SECTION 17: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 17: Notebook 23 Complete -- Handoff to Notebook 24")

print("NOTEBOOK 23: RISK TIER MONITORING -- COMPLETE")
print(f"  Champion model                    : {CHAMPION_NAME}")
print(f"  Monitoring windows (simulated)     : {N_MONITORING_WINDOWS}")
print(f"  Alerts / Watch                     : {_n_alerts} / {_n_watch}")
print(f"  Method agreement (kappa)           : {_kappa:.4f}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb23_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 24_risk_tier_reporting.ipynb")
print("\n\u2705 Ready to proceed.")
